# HerdNet Minimal working example

In [ ]:
# Install dependencies
!pip install \
    timm==1.0.22 \
    albumentations==1.0.3 \
    hydra-core==1.3.2 \
    opencv-python==4.10.0.84 \
    pillow==10.4.0 \
    scikit-image \
    scikit-learn \
    wandb==0.20.1 \
    gdown==5.2.0 \
    matplotlib==3.10.8 \
    tqdm==4.67.1 \
    loguru==0.7.3 \
    seaborn==0.13.2 \
    numpy==2.4.1 \
    scipy==1.16.0 \
    pandas==2.3.1

In [ ]:
## Installation

### Installation Colab

In [ ]:
# Download and install the code
import sys

!git clone -b dinov3 https://github.com/cwinkelmann/HerdNet
!cd '/content/HerdNet' && pip install .

sys.path.append('/content/HerdNet')

### Install using the conda environment

In [ ]:
# !conda env create -n HerdNet -f ../environment.yml

In [ ]:
## Optional update if the environment changed
!conda env update --file environment.yml --prune


## (Optional) Install Active Learning Repository
This contains functions for Training Data Preparation, Geospatial Inference and Detection Deduplication.
TODO

In [ ]:
import sys
sys.path.append('./')

In [ ]:
from pathlib import Path
Path("./").resolve()

# TODO

In [ ]:
from loguru import logger

logger.disable("animaloc")

### Train a model using checkppints

In [ ]:
from pathlib import Path
from hydra import initialize_config_dir, compose
from hydra.core.global_hydra import GlobalHydra
from omegaconf import OmegaConf

In [ ]:
## Create a config by hand:

# Create config
cfg = OmegaConf.create({
    "losses": {
        "FocalLoss": {
            "print_name": "focal_loss",
            "from_torch": False,
            "output_idx": 0,
            "target_idx": 0,
            "lambda_const": 1.0,
            "kwargs": {
                "alpha": 2,
                "beta": 4,
                "reduction": "mean",
                "normalize": False,
            }
        },
        "CrossEntropyLoss": {
            "print_name": "ce_loss",
            "from_torch": True,
            "output_idx": 1,
            "target_idx": 1,
            "lambda_const": 1.0,
            "background_class_weight": 0.1,
            "kwargs": {
                "reduction": "mean",
                "weight": [0.1, 5, 0.1],
            }
        }
    },
    "datasets": {
        "img_size": [512, 512],
        "anno_type": "point",
        "num_classes": 3,
        "collate_fn": None,
        "class_def": {
            1: "iguana_point",
            2: "hard_negative",
        },
        "train": {
            "name": "CSVDataset",
            "csv_file": None,
            "root_dir": None,
            "sampler": None,
        },
        "validate": {
            "name": "CSVDataset",
            "csv_file": None,
            "root_dir": None,
            "albu_transforms": {"Normalize": {"p": 1.0}},
            "end_transforms": {"DownSample": {"down_ratio": 4, "anno_type": "point"}},
        },
        "test": {
            "name": "CSVDataset",
            "csv_file": None,
            "root_dir": str(DATA_DIR),
            "albu_transforms": {"Normalize": {"p": 1.0}},
            "end_transforms": {"DownSample": {"down_ratio": 4, "anno_type": "point"}},
        },
    },
    "training_settings": {
        "trainer": "Trainer",
        "batch_size": 12,
        "num_workers": 2,
        "evaluator": {
            "name": "HerdNetEvaluator",
            "threshold": 100,
            "select_mode": "max",
            "validate_on": "f1_score",
            "kwargs": {
                "print_freq": 125,
                "lmds_kwargs": {
                    "kernel_size": [9, 9],
                    "adapt_ts": 0.3,
                    "scale_factor": 1,
                    "up": True,
                }
            }
        },
        "stitcher": {
            "name": "HerdNetStitcher",
            "kwargs": {
                "overlap": 120,
                "down_ratio": 4,
                "up": False,
                "reduction": "mean",
            }
        },
    },
    "model": {
        "name": "CamouflageHerdNetConvNeXt",
        "from_torchvision": False,
        "load_from": str(MODEL_PATH),
        "resume_from": None,
        "kwargs": {
            "pretrained": True,
            "down_ratio": 4,
            "backbone_size": "base",
        },
        "freeze": None,
    },
    "wandb_flag": False,
    "seed": 1,
    "device_name": None,  # auto-detect
})

In [ ]:


from animaloc.utils.train import main

# Clear any previous Hydra state (important in notebooks when re-running cells)
GlobalHydra.instance().clear()

# Configure paths
config_dir = str(Path.cwd() / "HerdNet/configs" / "demo")  # adjust as needed
config_name = "dla34_delplanque"

# Load config
with initialize_config_dir(config_dir=config_dir, version_base="1.1"):
    cfg = compose(config_name=config_name, overrides=[
        # Add any overrides here, e.g.:
        # "training.epochs=10",
        #  "training.batch_size=4",
        "model.load_from=/content/20220413_HerdNet_General_dataset_2022.pth"
    ])

# Inspect config (optional)
print(OmegaConf.to_yaml(cfg))



In [ ]:
# Run training
result = main(cfg)
wandb.finish()

### Inference with the model




In [ ]:
# Inference notebook cell

# Inference notebook cell

from pathlib import Path
from hydra import initialize_config_dir, compose
from hydra.core.global_hydra import GlobalHydra

from animaloc.utils.inference import inference

# Clear Hydra state
GlobalHydra.instance().clear()

config_dir = str(Path.cwd() / "configs" / "demo")  # adjust as needed
config_name = "dla34_delplanque"


# Load the just trained model and load a folder with images
with initialize_config_dir(config_dir=config_dir, version_base="1.1"):
    cfg = compose(config_name=config_name, overrides=[
        "model.load_from=/home/cwinkelmann/work/Herdnet/best_model.pth",
        "datasets.test.root_dir=/home/cwinkelmann/work/Herdnet/tests/data/single_images/ISWF01_22012023_subset",
        # add other overrides as needed
    ])

# Run inference
detections = inference(cfg, plain_inference=True, vis_detections=True)

# Show results
print(f"Total detections: {len(detections)}")
detections.head(10)

## Find optimal class weights to cope with class imbalance

In [ ]:
import yaml

class_counts = df['labels'].value_counts()

# 2. Convert counts to frequencies (i.e., fraction of the total)
class_freqs = class_counts / class_counts.sum()

# 3. Compute inverse frequency for each class
class_weights_inv = 1.0 / class_freqs

# 4. Convert to a dictionary {class_id: weight_value}
class_weights_dict = class_weights_inv.to_dict()

print("Class distribution:\n", class_freqs)
print("\nInverse frequency weights:\n", class_weights_dict)

class_weights_dict = class_weights_inv.to_dict()

# 5. Save to a YAML file
with open('class_weights.yaml', 'w') as f:
    yaml.safe_dump(class_weights_dict, f, sort_keys=True)